### Building a text generation pipeline


<div class="alert alert-block alert-info">
<b>1)</b> Use GPT-2 model for text completion
</div>

GPT-2 model card

https://github.com/openai/gpt-2/blob/master/model_card.md

Model date

February 2019, trained on data that cuts off at the end of 2017.

In [4]:
from transformers import pipeline, set_seed

In [116]:
# Set a random seed for reproducible results
set_seed(42)

# Load the pretrained GPT-2 text-generation model
gpt2_pipeline = pipeline(
                        "text-generation",
                        model="openai-community/gpt2"
                        )

# Define the starting prompt for text generation
prompt = "The Giraffe and the Pelly went to"

# Generate three text continuations
results = gpt2_pipeline(
                        prompt,
                        max_new_tokens=50,       # Generate up to 50 new tokens per result
                        num_return_sequences=5,   # Return five different continuations
                        do_sample=True,          # Sample tokens to introduce variation
                        temperature=0.7,          # Control randomness; lower values favour likely tokens
                        )

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 587.06it/s, Materializing param=transformer.wte.weight]             
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [117]:
# print the results
for result in results:
    print("=" * 60)
    print(result['generated_text'])

The Giraffe and the Pelly went to work on the game.

"I think it was really fun, I was really excited to play," said Latham. "I was really excited to get a chance to play with the Giraffes. They've been great. I was
The Giraffe and the Pelly went to the same place again. Once they had got there, the couple took a boat out and set up their tents in front of the house. They took care to make sure that no one would get in the way of their tent. The Giraffe then
The Giraffe and the Pelly went to the zoo to watch their zoo, which was in the zoo. They went to the zoo to watch their zoo, which was in the zoo. They went to the zoo to watch their zoo, which was in the zoo. They went to the zoo
The Giraffe and the Pelly went to a nearby field to find the carcass of his comrade. He was not in a good mood and he was being chased by a male wolf, so he was taken to a nearby pond.

The Giraffe immediately threw his comrade overboard, and
The Giraffe and the Pelly went to the front and the pokes were 

<div class="alert alert-block alert-info">
<b>2)</b> Loading datasets
</div>

### Loading datasets


In [121]:
from datasets import load_dataset

# Load the validation split of the TIGER-Lab/MMLU-Pro dataset
my_dataset = load_dataset(
    "TIGER-Lab/MMLU-Pro",
    split="validation"
)

# Display dataset details
print(my_dataset)

Dataset({
    features: ['question_id', 'question', 'options', 'answer', 'answer_index', 'cot_content', 'category', 'src'],
    num_rows: 70
})


### Manipulating datasets


In [ ]:
from datasets import load_dataset

# Load a Wikipedia dataset
wikipedia = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train"
)

In [123]:
# Filter the documents
filtered = wikipedia.filter(lambda row: "football" in row["text"])

# Create a sample dataset
example = filtered.select(range(1))

print(example[0]["text"])

Filter: 100%|██████████| 6407814/6407814 [02:10<00:00, 49221.46 examples/s]

Alabama () is a state in the Southeastern region of the United States, bordered by Tennessee to the north; Georgia to the east; Florida and the Gulf of Mexico to the south; and Mississippi to the west. Alabama is the 30th largest by area and the 24th-most populous of the U.S. states.

Alabama is nicknamed the Yellowhammer State, after the state bird. Alabama is also known as the "Heart of Dixie" and the "Cotton State". The state tree is the longleaf pine, and the state flower is the camellia. Alabama's capital is Montgomery, and its largest city by population and area is Huntsville. Its oldest city is Mobile, founded by French colonists (Alabama Creoles) in 1702 as the capital of French Louisiana. Greater Birmingham is Alabama's largest metropolitan area and its economic center.

Originally home to many native tribes, present-day Alabama was a Spanish territory beginning in the sixteenth century until the French acquired it in the early eighteenth century. The British won the territory

<div class="alert alert-block alert-info">
<b>3)</b> Grammatical correctness
</div>

In [125]:
# Create a pipeline for grammar checking

grammar_checker = pipeline(
                          task="text-classification", 
                          model="abdulmatinomotoso/English_Grammar_Checker"
                          )

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 662.32it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: abdulmatinomotoso/English_Grammar_Checker
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "c:\Users\rehan\anaconda3\envs\py3114\Lib\threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "c:\Users\rehan\anaconda3\envs\py3114\Lib\threading.py", line 975, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\rehan\anaconda3\envs\py3114\Lib\site-packages\transformers\safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "c:\Users\rehan\anaconda3\envs\p

    spawn_conversion(token, private, model_id)
  File "c:\Users\rehan\anaconda3\envs\py3114\Lib\site-packages\transformers\safetensors_conversion.py", line 49, in spawn_conversion
    event_id = result["event_id"]
               ~~~~~~^^^^^^^^^^^^
KeyError: 'event_id'


In [138]:
%pip install "huggingface_hub>=0.25.0,<2.0"

  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
Using cached huggingface_hub-1.33.0-py3-none-any.whl (846 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 2.2.0
    Uninstalling huggingface_hub-2.2.0:
      Successfully uninstalled huggingface_hub-2.2.0
Note: you may need to restart the kernel to use updated packages.


In [141]:
import transformers
import huggingface_hub
import datasets
import safetensors

print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("Datasets:", datasets.__version__)
print("SafeTensors:", safetensors.__version__)

Transformers: 5.2.0
Hugging Face Hub: 1.33.0
Datasets: 5.0.1
SafeTensors: 0.7.0


In [1]:
from transformers import pipeline

# Load the grammar acceptability classifier
grammar_checker = pipeline(
    task="text-classification",
    model="abdulmatinomotoso/English_Grammar_Checker"
)

c:\Users\rehan\anaconda3\envs\py3114\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 20100.50it/s]


In [3]:
# Check grammar of the input text

output = grammar_checker("He eat pizza every day")
print(output)

[{'label': 'LABEL_0', 'score': 0.9929908514022827}]


In [6]:
# Test examples with different grammatical quality
sentences = [
    "She goes to school every day.",  # Correct
    "She going to school every day.",    # Incorrect
    "They are playing football.",     # Correct
    "They is play football."       # Incorrect
]

# Classify each sentence
for sentence in sentences:
    result = grammar_checker(sentence)
    print(f"Sentence: {sentence}")
    print(f"Prediction: {result[0]['label']}")
    print(f"Score: {result[0]['score']:.4f}")
    print("-" * 50)

Sentence: She goes to school every day.
Prediction: LABEL_1
Score: 0.9997
--------------------------------------------------
Sentence: She going to school every day.
Prediction: LABEL_0
Score: 0.9969
--------------------------------------------------
Sentence: They are playing football.
Prediction: LABEL_1
Score: 0.9998
--------------------------------------------------
Sentence: They is play football.
Prediction: LABEL_0
Score: 0.9969
--------------------------------------------------


In [13]:
# Check grammar of the input text

output = grammar_checker("I picking my son")
print(output)

[{'label': 'LABEL_0', 'score': 0.9968698620796204}]


<div class="alert alert-block alert-info">
<b></b> Question Natural Language Inference
</div>

In [14]:
# Create the pipeline

classifier = pipeline(
                        task="text-classification",
                        model="cross-encoder/qnli-electra-base")



Loading weights: 100%|██████████| 201/201 [00:01<00:00, 166.85it/s]
c:\Users\rehan\anaconda3\envs\py3114\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rehan\.cache\huggingface\hub\models--cross-encoder--qnli-electra-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [19]:
# Predict the output
output = classifier(
                    "What is the capital of France?, Paris is capital of France."
                    )

print(output)

[{'label': 'LABEL_0', 'score': 0.997468113899231}]


In [20]:
# Predict the output
output = classifier("Where is the capital of France?, Brittany is known for its stunning coastline.")


print(output)

[{'label': 'LABEL_0', 'score': 0.016211964190006256}]


We have got two options here top use the classification of the questions and the answers

1. "cross-encoder/qnli-electra-base" which is from the hugging face and it is pre trained model. this was used just as an example to see how we can use the different models from huggingface.

2. "from sentence_transformers import CrossEncoder" is a python library which makes th emodelk easier to use.


In [24]:
# Import the CrossEncoder class
from sentence_transformers import CrossEncoder

# Load the pretrained QNLI model
model = CrossEncoder("cross-encoder/qnli-electra-base")

# Define the question and the passage separately
question = "Where is the capital of France?"
passage = "Brittany is known for its stunning coastline."

# Predict how well the passage answers the question
output = model.predict([(question, passage)])

# Display the result
print(output)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 243.77it/s]


[0.00395958]


In [25]:
# Define one question and two possible passages
question = "Where is the capital of France?"

passages = [
    "Paris is the capital of France.",
    "Brittany is known for its stunning coastline."
]

# Create question-passage pairs for the model
pairs = [(question, passage) for passage in passages]

# Calculate a relevance score for each pair
scores = model.predict(pairs)

# Print each passage alongside its score
for passage, score in zip(passages, scores):
    print(f"Passage: {passage}")
    print(f"Relevance score: {score:.4f}")
    print("-" * 40)

Passage: Paris is the capital of France.
Relevance score: 0.9983
----------------------------------------
Passage: Brittany is known for its stunning coastline.
Relevance score: 0.0040
----------------------------------------


<div class="alert alert-block alert-info">
<b></b> Dynamic category assignment
</div>

#### Use the bart-large-mnli model to classify the text into the categories which we can input

In [27]:
text = "AI-powered robots assist in complex brain surgeries with precision."

# Create the pipeline
classifier = pipeline(
                        task="zero-shot-classification", 
                        model="facebook/bart-large-mnli")

# Create the categories list
categories = ["science", "politics", "sports"]

# Predict the output
output = classifier(text, categories)

# Print the top label and its score
print(f"Top Label: {output['labels'][0]} with score: {output['scores'][0]}")

Loading weights: 100%|██████████| 515/515 [00:02<00:00, 248.85it/s]


Top Label: science with score: 0.9510334134101868


<div class="alert alert-block alert-info">
<b></b> Summarizing long text using facebook/bart-large-cnn
</div>

In [47]:
original_text = '''

Greece is a nation in southeastern Europe on the Balkan Peninsula, renowned as the cradle of Western civilization. 
The country features a mostly mountainous mainland alongside more than 6,000 islands and islets scattered across the Aegean and Ionian seas. 
Its capital, Athens, is a historic metropolis dominated by the ancient Acropolis and the Parthenon. With a warm Mediterranean climate, long coastlines, and a population of roughly 10.5 million people, Greece combines a dramatic natural landscape with a deeply rooted identity.

Ancient Greece left a monumental legacy that continues to shape modern society. 
It was the birthplace of democracy, Western philosophy, dramatic theater, and the Olympic Games. 
Influential thinkers like Socrates, Plato, and Aristotle lived and taught in city-states such as Athens and Sparta. 
Over the centuries, Greece experienced rule by the Roman and Byzantine empires, as well as centuries of Ottoman control, before fighting for and winning its independence in the 19th century.

Today, modern Greece is a democratic parliamentary republic and a member of the European Union. 
Its economy relies heavily on vibrant tourism, global shipping, and traditional agriculture—including olive oil and feta cheese production. 
Greek culture remains celebrated worldwide through its family-oriented traditions, Orthodox Christian heritage, 
lively music and dance like the Sirtaki, and a distinctive diet rich in fresh vegetables, fish, and olive oil.

'''

In [48]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "facebook/bart-large-cnn"
)

# Load the summarization model
model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/bart-large-cnn"
)

# Tokenize the original text
inputs = tokenizer(
    original_text,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

# Generate the summary
summary_ids = model.generate(
    **inputs,
    max_new_tokens=130,
    min_new_tokens=30,
    do_sample=False
)

# Decode the generated tokens
summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

# Compare character lengths
print(f"Original text length: {len(original_text)}")
print(f"Summary length: {len(summary)}")

print("\nSummary:")
print(summary)

Loading weights: 100%|██████████| 511/511 [00:02<00:00, 196.52it/s]
[transformers] Both `max_new_tokens` (=130) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=30) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original text length: 1489
Summary length: 343

Summary:
Greece is a nation in southeastern Europe on the Balkan Peninsula. Its capital, Athens, is a historic metropolis dominated by the ancient Acropolis and the Parthenon. With a warm Mediterranean climate, long coastlines, and a population of roughly 10.5 million people, Greece combines a dramatic natural landscape with a deeply rooted identity.


In [50]:
print(summary)

Greece is a nation in southeastern Europe on the Balkan Peninsula. Its capital, Athens, is a historic metropolis dominated by the ancient Acropolis and the Parthenon. With a warm Mediterranean climate, long coastlines, and a population of roughly 10.5 million people, Greece combines a dramatic natural landscape with a deeply rooted identity.


<div class="alert alert-block alert-info">
<b></b> Tokenizing text with AutoTokenizer
</div>

In [51]:
# Import necessary library for tokenization
from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

# Split input text into tokens
tokens = tokenizer.tokenize("AI: Making robots smarter and humans lazier!")

# Display the tokenized output
print(f"Tokenized output: {tokens}")

Tokenized output: ['ai', ':', 'making', 'robots', 'smarter', 'and', 'humans', 'la', '##zier', '!']


<div class="alert alert-block alert-info">
<b></b> Using AutoClasses
</div>

In [2]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

c:\Users\rehan\anaconda3\envs\py3114\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [55]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("Transformers path:", transformers.__file__)
print("PyTorch:", torch.__version__)

Transformers: 5.19.0
Transformers path: c:\Users\rehan\anaconda3\envs\py3114\Lib\site-packages\transformers\__init__.py
PyTorch: 2.12.1+cpu


In [5]:
# Download the model and tokenizer
my_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

my_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

# Create the pipeline
my_pipeline = pipeline(
                    task="sentiment-analysis",
                    model=my_model,
                    tokenizer=my_tokenizer)

# Predict the sentiment
output = my_pipeline("This course is pretty good, I guess.")
print(f"Sentiment using AutoClasses: {output[0]['label']}")

Device set to use cpu


Sentiment using AutoClasses: POSITIVE


### Extracting text with PyPDF

<div class="alert alert-block alert-info">
<b></b> Extract text from pdf of the highway code with PyPDF
</div>

In [7]:
from pypdf import PdfReader

# Extract text from the PDF
reader = PdfReader("the_official_highway_code_04-2025_3.pdf")

# Extract text from all pages
document_text = ""
for page in reader.pages: 
    document_text += page.extract_text()


In [9]:
print(document_text)


 highwaycodeuk.co.uk 
 Updated 10/04/2025 
 AAATHE 
 Contents 
 Introduction  3 
 Rules for pedestrians (1 to 35)  5 
 Rules for users of powered wheelchairs and mobility scooters (36 to 46)  19 
 Rules about animals (47 to 58)  23 
 Rules for cyclists (59 to 82)  28 
 Rules for motorcyclists (83 to 88)  38 
 Rules for drivers and motorcyclists (89 to 102)  44 
 General rules, techniques and advice for all drivers and riders (103 to 158)  59 
 Using the road (159 to 203)  8  4 
 Road users requiring extra care (204 to 225)  110 
 Driving in adverse weather conditions (226 to 237)  119 
 Waiting and parking (238 to 252)  125 
 Motorways (253 to 273)  132 
 Breakdowns and incidents (274 to 287)  140 
 Road works, level crossings and tramways (288 to 307)  147 
 Direction signs on roads and motorways  155 
 Information signs  155 
 Light signals controlling traffic  1  55 
 Road markings  1  55 
 Road signs giving orders  156 
 Road works signs  1  56 
 Signals by authorised persons  1  5

<div class="alert alert-block alert-info">
<b></b> Building a Q&A pipeline
</div>

In [ ]:
from pypdf import PdfReader

# Extract text from the PDF
reader = PdfReader("the_official_highway_code_04-2025_3.pdf")

# Extract text from all pages
document_text = ""
for page in reader.pages: 
    document_text += page.extract_text()


# Load the question-answering pipeline
qa_pipeline = pipeline(
                        task="question-answering",
                        model="distilbert-base-cased-distilled-squad"
                        )

question = "Where does HIghway Code applies to ?"

# Get the answer from the QA pipeline
result = qa_pipeline(
                    question=question,
                    context=document_text
                    )

# Print the answer
print(f"Answer: {result['answer']}")

Device set to use cpu


Answer: England, Scotland and Wales


In [12]:
def distilled_squad(question):


    # Load the question-answering pipeline
    qa_pipeline = pipeline(
                            task="question-answering",
                            model="distilbert-base-cased-distilled-squad"
                            )

    question = question

    # Get the answer from the QA pipeline
    result = qa_pipeline(
                        question=question,
                        context=document_text
                        )

    # Print the answer
    print(f"Answer: {result['answer']}")

In [ ]:
distilled_squad("Can I use handheld mobile phone while driving in England?")

Device set to use cpu


Answer: Never smoke


In [16]:
import re
from pypdf import PdfReader
from transformers import pipeline

# --------------------------------------------------
# 1. Extract text from the PDF
# --------------------------------------------------

reader = PdfReader("the_official_highway_code_04-2025_3.pdf")

pages_text = []

for page_number, page in enumerate(reader.pages, start=1):

    # Extract text from the current page
    text = page.extract_text() or ""

    # Normalise Windows and other line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove trailing spaces from each line
    text = "\n".join(line.strip() for line in text.splitlines())

    # Reduce excessive blank lines while preserving paragraphs
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Store the page number alongside its text
    if text.strip():
        pages_text.append(f"\n[PAGE {page_number}]\n{text}")

# Combine pages into one document
document_text = "\n\n".join(pages_text)

# --------------------------------------------------
# 2. Check the extracted text
# --------------------------------------------------

print("Number of pages:", len(reader.pages))
print("Extracted characters:", len(document_text))
print("\nFirst 2,000 characters:\n")
print(document_text[:2000])

# --------------------------------------------------
# 3. Load the question-answering model
# --------------------------------------------------

qa_pipeline = pipeline(
    task="question-answering",
    model="distilbert/distilbert-base-cased-distilled-squad"
)

# --------------------------------------------------
# 4. Ask a question
# --------------------------------------------------

question = "Can I use handheld mobile phone while driving in England?"

result = qa_pipeline(
    question=question,
    context=document_text,
    max_seq_len=384,
    doc_stride=128
)

# --------------------------------------------------
# 5. Display the answer and confidence score
# --------------------------------------------------

print("\nQuestion:", question)
print("Answer:", result["answer"])
print("Score:", round(result["score"], 4))

Number of pages: 236
Extracted characters: 236178

First 2,000 characters:


[PAGE 1]
highwaycodeuk.co.uk
Updated 10/04/2025


[PAGE 2]
AAATHE
Contents
Introduction  3
Rules for pedestrians (1 to 35)  5
Rules for users of powered wheelchairs and mobility scooters (36 to 46)  19
Rules about animals (47 to 58)  23
Rules for cyclists (59 to 82)  28
Rules for motorcyclists (83 to 88)  38
Rules for drivers and motorcyclists (89 to 102)  44
General rules, techniques and advice for all drivers and riders (103 to 158)  59
Using the road (159 to 203)  8  4
Road users requiring extra care (204 to 225)  110
Driving in adverse weather conditions (226 to 237)  119
Waiting and parking (238 to 252)  125
Motorways (253 to 273)  132
Breakdowns and incidents (274 to 287)  140
Road works, level crossings and tramways (288 to 307)  147
Direction signs on roads and motorways  155
Information signs  155
Light signals controlling traffic  1  55
Road markings  1  55
Road signs giving orders  156
Road works si

c:\Users\rehan\anaconda3\envs\py3114\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rehan\.cache\huggingface\hub\models--distilbert--distilbert-base-cased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Device set to use cpu



Question: Can I use handheld mobile phone while driving in England?
Answer: ●  You  MUST NOT  do anything illegal
Score: 0.2405


In [17]:
from transformers import AutoTokenizer

model_name = "distilbert/distilbert-base-cased-distilled-squad"

tokenizer = AutoTokenizer.from_pretrained(model_name)

question = "Where does the Highway Code apply?"

# Tokenise the question and context as a pair
inputs = tokenizer(
    question,
    document_text,
    truncation="only_second",
    max_length=384,
    return_tensors="pt"
)

print("Input tensor shape:", inputs["input_ids"].shape)
print("Number of input tokens:", inputs["input_ids"].shape[1])
print("\nDecoded model input:\n")
print(tokenizer.decode(inputs["input_ids"][0]))

Input tensor shape: torch.Size([1, 384])
Number of input tokens: 384

Decoded model input:

[CLS] Where does the Highway Code apply? [SEP] [ PAGE 1 ] highwaycodeuk. co. uk Updated 10 / 04 / 2025 [ PAGE 2 ] AAATHE Contents Introduction 3 Rules for pedestrians ( 1 to 35 ) 5 Rules for users of powered wheelchairs and mobility scooters ( 36 to 46 ) 19 Rules about animals ( 47 to 58 ) 23 Rules for cyclists ( 59 to 82 ) 28 Rules for motorcyclists ( 83 to 88 ) 38 Rules for drivers and motorcyclists ( 89 to 102 ) 44 General rules, techniques and advice for all drivers and riders ( 103 to 158 ) 59 Using the road ( 159 to 203 ) 8 4 Road users requiring extra care ( 204 to 225 ) 110 Driving in adverse weather conditions ( 226 to 237 ) 119 Waiting and parking ( 238 to 252 ) 125 Motorways ( 253 to 273 ) 132 Breakdowns and incidents ( 274 to 287 ) 140 Road works, level crossings and tramways ( 288 to 307 ) 147 Direction signs on roads and motorways 155 Information signs 155 Light signals controlling